# 09 · GenAI Engineering Assistant (local RAG + text-to-SQL)

An assistant that answers engineers' questions about the 70 test drives using **only this project's data**,
running fully **on the laptop** with no cloud calls, so no data leaves the machine.

```
question ──► router (LLM, JSON) ──┬── "docs"  ─► retrieve report passages ─► LLM answer with [citations]
                                  ├── "sql"   ─► LLM writes SQL ─► read-only DuckDB views ─► LLM answer from the rows
                                  └── "out_of_scope" ─► polite refusal
```

| Part | Choice | Why |
|---|---|---|
| LLM | `gemma2:2b` via Ollama (about 1.6 GB, runs comfortably on a 6 GB GPU) | small, fast, local; its limits are measured honestly in the evaluation |
| Embeddings | `nomic-embed-text` via Ollama | small, local, good retrieval quality |
| Knowledge base | 70 trip reports + project findings, generated from the warehouse | every fact traces back to verified data |
| Vector store | DuckDB table `assistant.chunks` | same warehouse, nothing extra to run |
| Numbers | text-to-SQL on 5 curated, read-only views | numbers come from the database, not from the LLM's memory |

**Evaluation:** about 30 questions with correct answers computed directly from the warehouse. Each is scored
automatically for answer correctness, routing and refusal of out-of-scope questions.

In [1]:
from pathlib import Path
import json
import re
import time

import duckdb
import nbformat
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "warehouse" / "volvo.duckdb"
NOTEBOOKS = ROOT / "notebooks"
REPORTS = ROOT / "reports"
KB_DIR = ROOT / "knowledge_base"
KB_DIR.mkdir(exist_ok=True)
OLLAMA = "http://127.0.0.1:11434"
LLM = "gemma2:2b"
EMBED = "nomic-embed-text"
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)

installed = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags", timeout=10).json()["models"]]
print("Ollama models:", installed)
assert any(m.startswith(LLM) for m in installed) and any(m.startswith(EMBED) for m in installed)

Ollama models: ['gemma2:2b', 'nomic-embed-text:latest']


## 1 · Read-only views for questions about numbers

In [2]:
con = duckdb.connect(str(DB_PATH))
con.execute("DROP SCHEMA IF EXISTS assistant CASCADE")
con.execute("CREATE SCHEMA assistant")
con.execute("""
CREATE VIEW assistant.v_trips AS
WITH sg AS (SELECT trip_id, count(*) AS sag_events, median(apparent_resistance_mohm) AS median_resistance_mohm
            FROM signal.sag_events GROUP BY 1),
     fl AS (SELECT trip_id, count(*) AS anomaly_flags FROM anomaly.flags GROUP BY 1),
     dr AS (SELECT c.trip_id, avg(c.ambient_temp_c) AS ambient_mean_c, avg(c.speed_kmh) AS mean_driving_speed_kmh
            FROM clean.trip_signals c JOIN signal.samples s USING (trip_id, row_idx) WHERE s.phase = 'driving' GROUP BY 1)
SELECT t.trip_id, t.season, t.route, t.weather, CAST(t.start_time AS DATE) AS trip_date,
       round(e.distance_km_integrated, 2) AS distance_km, round(t.duration_min, 1) AS duration_min,
       round(dr.ambient_mean_c, 1) AS ambient_mean_c, round(dr.mean_driving_speed_kmh, 1) AS mean_driving_speed_kmh,
       r.batt_temp_start_c, r.rise_c AS batt_temp_rise_c,
       round(e.consumption_kwh_per_100km, 2) AS consumption_kwh_per_100km,
       round(e.energy_out_kwh, 3) AS energy_out_kwh, round(e.energy_regen_kwh, 3) AS energy_regen_kwh,
       round(e.heating_kwh, 3) AS heating_kwh, round(e.energy_charged_kwh, 3) AS energy_charged_kwh,
       round(e.heating_share_pct, 1) AS heating_share_pct, round(e.regen_share_pct, 1) AS regen_share_pct,
       t.driving_min, t.stopped_min, t.charging_min,
       coalesce(sg.sag_events, 0) AS sag_events, round(sg.median_resistance_mohm, 1) AS median_resistance_mohm,
       coalesce(fl.anomaly_flags, 0) AS anomaly_flags,
       t.trip_id IN (SELECT trip_id FROM model.test_trips) AS is_test_trip
FROM clean.trips t JOIN signal.trip_energy e USING (trip_id) JOIN signal.temp_rise r USING (trip_id)
JOIN dr USING (trip_id) LEFT JOIN sg USING (trip_id) LEFT JOIN fl USING (trip_id)
""")
con.execute("""CREATE VIEW assistant.v_anomalies AS
SELECT flag_id, detector, level, trip_id, round(score, 2) AS score, reason FROM anomaly.flags""")
con.execute("""CREATE VIEW assistant.v_sag_events AS
SELECT trip_id, round(start_s, 1) AS start_s, batt_temp_c, round(soc_pct, 1) AS soc_pct,
       round(-current_at_max_sag_a, 1) AS discharge_current_a, round(max_sag_v, 1) AS sag_v,
       round(apparent_resistance_mohm, 1) AS resistance_mohm, round(expected_resistance_mohm, 1) AS expected_resistance_mohm,
       flag AS flagged
FROM anomaly.resistance_events""")
con.execute("""CREATE VIEW assistant.v_model_results AS
SELECT task, split, stage, model, round(mae, 3) AS mae, round(rmse, 3) AS rmse, round(r2, 3) AS r2, is_selected
FROM model.metrics""")
con.execute("""CREATE VIEW assistant.v_soc_test_trips AS
SELECT trip_id, season, round(avg(abs_error), 2) AS mean_abs_soc_error_pts, count(*) AS samples
FROM model.soc_test_predictions GROUP BY 1, 2""")
for v in ["v_trips", "v_anomalies", "v_sag_events", "v_model_results", "v_soc_test_trips"]:
    print(v, con.sql(f"SELECT count(*) FROM assistant.{v}").fetchone()[0], "rows")

v_trips 70 rows
v_anomalies 40 rows
v_sag_events 556 rows
v_model_results 42 rows
v_soc_test_trips 14 rows


## 2 · Knowledge base: generated from the warehouse, never typed by hand

- **Trip reports:** one per trip, with energy, climate load, battery behaviour, phases, data-quality notes and every anomaly flag.
- **Project findings:** the findings and summary sections of notebooks 02–06, split by heading.
- **Cleaning log and signal dictionary.**

All documents are written to `knowledge_base/` so they can be read, and searched, outside the assistant too.

In [3]:
trips = con.sql("SELECT * FROM assistant.v_trips ORDER BY trip_id").df()
flags = con.sql("SELECT * FROM assistant.v_anomalies ORDER BY trip_id, flag_id").df()
soc_err = con.sql("SELECT * FROM assistant.v_soc_test_trips").df().set_index("trip_id")
gaps = con.sql("SELECT trip_id, avg(soc_gap::INT) AS gap, avg(heating_can_invalid::INT) AS stale FROM clean.trip_signals GROUP BY 1").df().set_index("trip_id")


def trip_report(r):
    t = r.trip_id
    lines = [f"# Trip report {t}",
             f"{t} is a {r.season} trip on {str(r.trip_date)[:10]} ({r.route}, weather: {r.weather}). "
             f"It covered {r.distance_km:.1f} km in {r.duration_min:.0f} minutes: {r.driving_min:.0f} min driving, "
             f"{r.stopped_min:.0f} min stopped" + (f", {r.charging_min:.0f} min charging while parked" if r.charging_min > 0 else "") + ".",
             f"Mean outside temperature while driving was {r.ambient_mean_c:.1f} °C. The battery started at "
             f"{r.batt_temp_start_c:.0f} °C and warmed by {r.batt_temp_rise_c:.0f} °C.",
             f"Energy: consumption {r.consumption_kwh_per_100km:.1f} kWh per 100 km; {r.energy_out_kwh:.2f} kWh drawn from the battery, "
             f"{r.energy_regen_kwh:.2f} kWh recovered by regenerative braking ({r.regen_share_pct:.0f} % of drawn energy), "
             f"{r.heating_kwh:.2f} kWh used for cabin heating ({r.heating_share_pct:.0f} % of drawn energy)."
             + (f" {r.energy_charged_kwh:.2f} kWh were charged into the battery." if r.energy_charged_kwh > 0 else ""),
             f"Battery: {r.sag_events} heavy-load voltage-sag events" +
             (f", median apparent pack resistance {r.median_resistance_mohm:.0f} mΩ." if r.sag_events else ".")]
    g = gaps.loc[t]
    if g.gap > 0:
        lines.append(f"Data quality: SoC was not logged for the last {g.gap:.0%} of this trip.")
    if g.stale > 0.05:
        lines.append(f"Data quality: the CAN heating-power signal was stale in {g.stale:.0%} of samples "
                     "(heating reported while none was requested); those values were removed.")
    if t in soc_err.index:
        lines.append(f"This trip was a locked test trip for the SoC model; the model's mean error on it was "
                     f"{soc_err.mean_abs_soc_error_pts[t]:.1f} SoC points.")
    f = flags[flags.trip_id == t]
    if len(f):
        lines.append(f"Anomaly flags ({len(f)}):")
        lines += [f"- {d.detector}: {d.reason}" for d in f.itertuples()]
    else:
        lines.append("Anomaly flags: none.")
    return "\n".join(lines)


docs = []
for r in trips.itertuples():
    text = trip_report(r)
    (KB_DIR / f"trip_{r.trip_id}.md").write_text(text, encoding="utf-8")
    docs.append(dict(doc_id=f"trip_{r.trip_id}", source=f"trip report {r.trip_id}", trip_id=r.trip_id, text=text))
print(docs[0]["text"])

# Trip report TripA01
TripA01 is a summer trip on 2019-06-25 (Munich East, weather: sunny). It covered 7.4 km in 17 minutes: 15 min driving, 2 min stopped.
Mean outside temperature while driving was 30.7 °C. The battery started at 21 °C and warmed by 1 °C.
Energy: consumption 17.4 kWh per 100 km; 1.67 kWh drawn from the battery, 0.38 kWh recovered by regenerative braking (23 % of drawn energy), 0.00 kWh used for cabin heating (0 % of drawn energy).
Battery: 0 heavy-load voltage-sag events.
Anomaly flags: none.


In [4]:
def notebook_sections(path, wanted):
    nb = nbformat.read(path, as_version=4)
    out = []
    for c in nb.cells:
        if c.cell_type == "markdown" and any(c.source.lstrip("# ").lower().startswith(w) for w in wanted):
            out.append(c.source)
    return "\n\n".join(out)


SECTIONS = {
    "02_data_preparation.ipynb": ["9 · summary", "3.3", "3.5", "3.2"],
    "03_signal_processing.ipynb": ["8 · summary", "**reading the cross-checks**"],
    "04_eda.ipynb": ["7 · findings"],
    "05_predictive_modelling.ipynb": ["6 · findings"],
    "06_anomaly_detection.ipynb": ["5 · findings"],
}
for name, wanted in SECTIONS.items():
    text = notebook_sections(NOTEBOOKS / name, [w.lower() for w in wanted])
    assert text, name
    (KB_DIR / f"findings_{name.replace('.ipynb', '')}.md").write_text(text, encoding="utf-8")
    # split into passages at bold paragraph starts and headings, about 900 characters max
    parts, buf = [], ""
    for para in re.split(r"\n\s*\n", text):
        if buf and (len(buf) + len(para) > 900 or para.startswith(("**", "#"))):
            parts.append(buf.strip())
            buf = ""
        buf += para + "\n\n"
    if buf.strip():
        parts.append(buf.strip())
    for i, p in enumerate(parts, start=1):
        docs.append(dict(doc_id=f"{name[:2]}_{i}", source=f"notebook {name[:-6]}", trip_id=None, text=p))

log = pd.read_csv(REPORTS / "02_cleaning_log.csv")
log_text = "# Data cleaning log\n" + "\n".join(f"- {r.issue}: {r.action} (trips: {r.trips}; values changed: {r.values_changed})"
                                              for r in log.itertuples())
dictionary = pd.read_csv(REPORTS / "01_signal_dictionary.csv")
dict_text = "# Signal dictionary\n" + "\n".join(f"- {r.signal} [{r.unit}] ({r.group}; summer trips {r.in_A_trips}/32, "
                                                 f"winter trips {r.in_B_trips}/38)" for r in dictionary.itertuples() if r.signal)
for doc_id, source, text in [("cleaning_log", "cleaning log", log_text), ("signal_dictionary", "signal dictionary", dict_text)]:
    (KB_DIR / f"{doc_id}.md").write_text(text, encoding="utf-8")
    docs.append(dict(doc_id=doc_id, source=source, trip_id=None, text=text))

kb = pd.DataFrame(docs)
print(f"knowledge base: {len(kb)} passages | {kb.text.str.len().sum():,} characters")
kb.source.str.split(" ").str[0].value_counts()

knowledge base: 106 passages | 66,097 characters


source
trip        70
notebook    34
cleaning     1
signal       1
Name: count, dtype: int64

## 3 · Embeddings and retrieval

In [5]:
def embed(texts, kind):
    # nomic-embed-text expects a task prefix
    prefix = "search_document: " if kind == "doc" else "search_query: "
    out = []
    for i in range(0, len(texts), 16):
        r = requests.post(f"{OLLAMA}/api/embed", json={"model": EMBED, "input": [prefix + t for t in texts[i:i + 16]]},
                          timeout=300)
        r.raise_for_status()
        out += r.json()["embeddings"]
    v = np.array(out, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


t0 = time.time()
E = embed(kb.text.tolist(), "doc")
print(f"embedded {len(kb)} passages in {time.time() - t0:.1f} s, dimension {E.shape[1]}")
kb_store = kb.assign(embedding=list(E))
con.register("frame_view", kb_store)
con.execute("CREATE TABLE assistant.chunks AS SELECT * FROM frame_view")
con.unregister("frame_view")
con.close()  # from here on the assistant only reads, through a read-only connection
ro = duckdb.connect(str(DB_PATH), read_only=True)
VIEW_COLUMNS = {v: set(ro.sql(f"SELECT * FROM assistant.{v} LIMIT 0").columns)
                for v in ["v_trips", "v_anomalies", "v_sag_events", "v_model_results", "v_soc_test_trips"]}

TRIP_RE = re.compile(r"\b(?:[Tt]rip\s*([AaBb])\s*(\d{1,2})|([AB])(\d{2}))\b")


def mentioned_trips(q):
    found = set()
    for m in TRIP_RE.finditer(q):
        letter, num = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
        if 1 <= int(num) <= 38:
            found.add(f"Trip{letter.upper()}{int(num):02d}")
    return sorted(found)


def retrieve(question, k=5):
    scores = E @ embed([question], "query")[0]
    order = list(np.argsort(-scores)[:k])
    # a question that names a trip always gets that trip's report, ranked first
    for t in mentioned_trips(question):
        idx = kb.index[kb.doc_id == f"trip_{t}"]
        if len(idx):
            order = [idx[0]] + [i for i in order if i != idx[0]][: k - 1]
    return kb.loc[order].assign(score=scores[order])


retrieve("Why does winter consumption rise?")[["doc_id", "source", "score"]]

embedded 106 passages in 1.6 s, dimension 768


,doc_id,source,score
81,04_2,notebook 04_eda,0.680525
82,04_3,notebook 04_eda,0.645593
83,04_4,notebook 04_eda,0.638497
84,04_5,notebook 04_eda,0.607778
101,06_5,notebook 06_anomaly_detection,0.602659


## 4 · The LLM: router, document answers and text-to-SQL

In [6]:
def chat(messages, json_mode=False, temperature=0.0):
    payload = {"model": LLM, "messages": messages, "stream": False,
               "options": {"temperature": temperature, "num_ctx": 8192, "seed": 7}}
    if json_mode:
        payload["format"] = "json"
    r = requests.post(f"{OLLAMA}/api/chat", json=payload, timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]


ROUTER = """You route questions for an assistant that knows ONLY one dataset: 70 recorded test drives of a BMW i3
electric car around Munich (32 summer trips 2019, 38 winter trips 2019/20), with battery voltage, current, temperature,
state of charge (SoC), heating, energy per trip, voltage-sag events, machine-learning models and anomaly flags.

Return JSON {"route": "..."} with exactly one of:
- "sql": the answer is a number, count, ranking, maximum/minimum, average or list of trips that can be computed from per-trip
  data (e.g. "Which trip had the highest consumption?", "How many winter trips?", "Average heating share in winter?").
- "docs": the answer is an explanation, a reason or a method (why/how questions, why a trip was flagged, how data was
  cleaned, which model was chosen and why, what was found).
- "out_of_scope": anything not answerable from this dataset (other cars, prices, the future, live or current data,
  people, general knowledge).

Examples:
"Which summer trip was the shortest?" -> {"route": "sql"}
"How many trips had more than 10 sag events?" -> {"route": "sql"}
"Which trip has the fewest anomaly flags?" -> {"route": "sql"}
"What is the mean consumption of winter trips?" -> {"route": "sql"}
"Why was TripA07 flagged as unusual?" -> {"route": "docs"}
"How was the coolant-flow signal converted?" -> {"route": "docs"}
"Why is the SoC model worse in the cold?" -> {"route": "docs"}
"Who owns the test car?" -> {"route": "out_of_scope"}
"What is the battery temperature at this moment?" -> {"route": "out_of_scope"}
"How fast does a Nissan Leaf charge?" -> {"route": "out_of_scope"}"""


def route(question):
    try:
        r = json.loads(chat([{"role": "system", "content": ROUTER}, {"role": "user", "content": question}], json_mode=True))
        return r.get("route", "docs") if r.get("route") in ("sql", "docs", "out_of_scope") else "docs"
    except (json.JSONDecodeError, requests.RequestException):
        return "docs"


DOC_PROMPT = """You are an engineering assistant for a battery test-drive dataset. Answer the question using ONLY the
context passages. Rules:
- Be concise (2-4 sentences) and specific; quote the numbers from the context.
- After each fact, cite the passage id in square brackets, e.g. [trip_TripB25] or [04_2].
- Give the reason or conclusion stated in the context, not only the observation (e.g. say WHY, and whether it is real).
- Never invent numbers; use only numbers that appear in the context.
- Questions about live or current values, people, other vehicles or the future are never in the context.
- If the context does not contain the answer, reply exactly: Not in the data: I can't answer that from this project's data.
"""


def answer_docs(question):
    ctx = retrieve(question)
    context = "\n\n".join(f"[{r.doc_id}] ({r.source})\n{r.text}" for r in ctx.itertuples())
    text = chat([{"role": "system", "content": DOC_PROMPT},
                 {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    return text, ctx.doc_id.tolist()

In [7]:
SCHEMA = """DuckDB views (read-only). Season values are 'summer' and 'winter'. Trip ids look like 'TripA05', 'TripB25'.
assistant.v_trips(one row per trip): trip_id, season, route, weather, trip_date, distance_km, duration_min, ambient_mean_c
  (mean outside temperature while driving), mean_driving_speed_kmh, batt_temp_start_c, batt_temp_rise_c,
  consumption_kwh_per_100km, energy_out_kwh, energy_regen_kwh, heating_kwh, energy_charged_kwh, heating_share_pct
  (heating energy as % of drawn energy), regen_share_pct, driving_min, stopped_min, charging_min (minutes charging while
  parked), sag_events (heavy-load voltage-sag events), median_resistance_mohm, anomaly_flags (count), is_test_trip (boolean)
assistant.v_anomalies(one row per flag): flag_id, detector, level, trip_id, score, reason
assistant.v_sag_events(one row per voltage-sag event): trip_id, start_s, batt_temp_c, soc_pct, discharge_current_a, sag_v,
  resistance_mohm, expected_resistance_mohm, flagged (boolean)
assistant.v_model_results(model benchmark): task ('A · SoC estimation' or 'B · segment energy'), split ('cross-validation' or
  'test'), stage, model, mae, rmse, r2, is_selected (boolean: the final chosen model)
assistant.v_soc_test_trips(SoC model error per locked test trip): trip_id, season, mean_abs_soc_error_pts, samples"""

SQL_PROMPT = f"""Write ONE DuckDB SQL query that answers the question. {SCHEMA}
Rules:
- a single SELECT (WITH allowed), only these views, no semicolon; no WHERE filter the question does not ask for
- per-trip questions (counts of trips, totals, averages, top/bottom trips) use assistant.v_trips
- lowest / coldest / fewest / smallest -> ORDER BY ... ASC; highest / warmest / most / largest -> ORDER BY ... DESC
- include trip_id and the value used for ranking when the question is about trips; use LIMIT for top/bottom questions
- counting trips: SELECT count(*) ... ; never mix aggregates with plain columns without GROUP BY
Examples:
Q: Which summer trip was the shortest? -> {{"sql": "SELECT trip_id, distance_km FROM assistant.v_trips WHERE season = 'summer' ORDER BY distance_km ASC LIMIT 1"}}
Q: How many summer trips had a mean outside temperature above 25 °C? -> {{"sql": "SELECT count(*) AS trips FROM assistant.v_trips WHERE season = 'summer' AND ambient_mean_c > 25"}}
Q: What is the average regeneration share in summer trips? -> {{"sql": "SELECT round(avg(regen_share_pct), 1) AS avg_regen_share_pct FROM assistant.v_trips WHERE season = 'summer'"}}
Q: Which trip has the fewest voltage-sag events? -> {{"sql": "SELECT trip_id, sag_events FROM assistant.v_trips ORDER BY sag_events ASC LIMIT 1"}}
Q: How many sag events happened with the battery below 0 °C? -> {{"sql": "SELECT count(*) AS events FROM assistant.v_sag_events WHERE batt_temp_c < 0"}}
Q: What is the test error of the selected segment-energy model? -> {{"sql": "SELECT model, mae, r2 FROM assistant.v_model_results WHERE task = 'B · segment energy' AND split = 'test' AND is_selected"}}
Return JSON {{"sql": "..."}}."""

ALLOWED = {"v_trips", "v_anomalies", "v_sag_events", "v_model_results", "v_soc_test_trips"}
FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|create|alter|attach|copy|pragma|install|load|export|call|set)\b", re.I)
def safe_sql(sql):
    sql = sql.strip().rstrip(";")
    if ";" in sql or FORBIDDEN.search(sql) or not re.match(r"^\s*(select|with)\b", sql, re.I):
        raise ValueError("only a single SELECT is allowed")
    used = set(re.findall(r"assistant\.(\w+)", sql))
    if not used or not used <= ALLOWED or re.search(r"\b(clean|raw|signal|model|anomaly|meta)\.", sql):
        raise ValueError("only the assistant views may be queried")
    return f"SELECT * FROM ({sql}) AS q LIMIT 50"


def answer_sql(question):
    messages = [{"role": "system", "content": SQL_PROMPT}, {"role": "user", "content": question}]
    sql, rows, error = None, None, None
    for attempt in range(2):  # one retry with the error message
        try:
            sql = json.loads(chat(messages, json_mode=True))["sql"]
            rows = ro.sql(safe_sql(sql)).df()
            break
        except Exception as e:  # noqa: BLE001 - any failure is fed back to the model once
            error = str(e)[:300]
            hint = ""
            missing = re.search(r'column "?(\w+)"? not found', error, re.I)
            if missing:  # schema linking: point the model at the view that really has this column
                homes = [v for v, cols in VIEW_COLUMNS.items() if missing.group(1) in cols]
                if homes:
                    hint = f" The column {missing.group(1)} exists in: {', '.join('assistant.' + h for h in homes)}."
            messages += [{"role": "assistant", "content": json.dumps({"sql": sql})},
                         {"role": "user", "content": f"That failed with: {error}.{hint} Return a corrected query as JSON."}]
    if rows is None:
        return f"Not in the data: the database query failed ({error}).", sql
    text = chat([{"role": "system", "content": "Answer the question in 1-3 sentences using ONLY the query result. "
                  "Quote the exact numbers and trip ids from the result and never add numbers that are not in it. "
                  "If the result is empty, say so."},
                 {"role": "user", "content": f"Question: {question}\nSQL: {sql}\nResult:\n{rows.head(20).to_string(index=False)}"}])
    return text, sql


REFUSAL = "Not in the data: this assistant only answers questions about the 70 BMW i3 test drives in this project."


def ask(question):
    t0 = time.time()
    r = route(question)
    if r == "sql":
        text, evidence = answer_sql(question)
    elif r == "docs":
        text, evidence = answer_docs(question)
    else:
        text, evidence = REFUSAL, None
    return dict(question=question, route=r, answer=text, evidence=evidence, seconds=round(time.time() - t0, 1))


demo = ask("Why was TripB25 flagged?")
print(demo["route"], "|", demo["seconds"], "s\n", demo["answer"], "\n", demo["evidence"])

docs | 1.1 s
 TripB25 was flagged for anomalies related to battery resistance.  The battery resistance was higher than expected, and this was flagged in the anomaly detection process. [06_2] 
 
 ['trip_TripB25', '06_2', 'cleaning_log', '02_2', '06_6']


## 5 · Evaluation set

The correct answer to every question is computed **directly from the warehouse**, independently of the assistant.
Scoring rules:
- **Numbers and trips:** the answer must contain the expected trip id(s) or number (rounded, within a small tolerance).
- **Explanations:** the answer must mention the expected key terms.
- **Out of scope:** the answer must refuse ("Not in the data").

Routing is scored separately, so a correct answer reached by an unexpected route still counts as correct.

In [8]:
q = lambda sql: ro.sql(sql).fetchall()
v = "assistant.v_trips"
top_cons = q(f"SELECT trip_id FROM {v} ORDER BY consumption_kwh_per_100km DESC LIMIT 1")[0][0]
low_cons = q(f"SELECT trip_id FROM {v} ORDER BY consumption_kwh_per_100km LIMIT 1")[0][0]
longest = q(f"SELECT trip_id FROM {v} ORDER BY distance_km DESC LIMIT 1")[0][0]
coldest = q(f"SELECT trip_id, ambient_mean_c FROM {v} ORDER BY ambient_mean_c LIMIT 1")[0]
top_sag = [r[0] for r in q(f"SELECT trip_id FROM {v} ORDER BY sag_events DESC LIMIT 3")]
charging = [r[0] for r in q(f"SELECT trip_id FROM {v} WHERE charging_min > 0 ORDER BY 1")]
worst_soc = q("SELECT trip_id FROM assistant.v_soc_test_trips ORDER BY mean_abs_soc_error_pts DESC LIMIT 1")[0][0]
most_flags = q(f"SELECT trip_id, anomaly_flags FROM {v} ORDER BY anomaly_flags DESC, trip_id LIMIT 1")[0]
n_winter = q(f"SELECT count(*) FROM {v} WHERE season = 'winter'")[0][0]
n_cold_start = q(f"SELECT count(*) FROM {v} WHERE batt_temp_start_c < 5")[0][0]
winter_km = q(f"SELECT round(sum(distance_km), 1) FROM {v} WHERE season = 'winter'")[0][0]
med_summer = q(f"SELECT round(median(consumption_kwh_per_100km), 1) FROM {v} WHERE season = 'summer'")[0][0]
heat_winter = q(f"SELECT round(avg(heating_share_pct), 1) FROM {v} WHERE season = 'winter'")[0][0]
b14_flags = q(f"SELECT anomaly_flags FROM {v} WHERE trip_id = 'TripB14'")[0][0]
n_flagged_events = q("SELECT count(*) FROM assistant.v_sag_events WHERE flagged")[0][0]
soc_mae = q("SELECT round(mae, 2) FROM assistant.v_model_results WHERE task LIKE 'A%' AND split = 'test' AND is_selected")[0][0]

EVAL = [
    # (question, category, expected route, check type, expected)
    ("Which trip had the highest energy consumption per 100 km?", "numbers", "sql", "ids", [top_cons]),
    ("Which trip had the lowest consumption per 100 km?", "numbers", "sql", "ids", [low_cons]),
    ("Which trip covered the longest distance?", "numbers", "sql", "ids", [longest]),
    ("How many winter trips are in the dataset?", "numbers", "sql", "count", n_winter),
    ("Which trip had the coldest mean outside temperature?", "numbers", "sql", "ids", [coldest[0]]),
    ("Which three trips had the most voltage-sag events?", "numbers", "sql", "ids", top_sag),
    ("Which trips include charging while parked?", "numbers", "sql", "ids", charging),
    ("What was the median consumption of summer trips in kWh per 100 km?", "numbers", "sql", "num", med_summer),
    ("What is the average heating share of energy in winter trips?", "numbers", "sql", "num", heat_winter),
    ("How many trips started with the battery colder than 5 °C?", "numbers", "sql", "count", n_cold_start),
    ("How many kilometres were driven in winter in total?", "numbers", "sql", "num", winter_km),
    ("How many anomaly flags does TripB14 have?", "numbers", "sql", "count", b14_flags),
    ("Which trip has the most anomaly flags?", "numbers", "sql", "ids", [most_flags[0]]),
    ("Which test trip had the largest SoC model error?", "numbers", "sql", "ids", [worst_soc]),
    ("How many voltage-sag events were flagged as anomalous?", "numbers", "sql", "count", n_flagged_events),
    ("What is the test-set error of the selected SoC model?", "numbers", "sql", "num", soc_mae),
    ("Why was TripB25 flagged?", "explain", "docs", "terms", {"all": ["resistance"], "any": ["soc", "state of charge"]}),
    ("Why does energy consumption rise in winter?", "explain", "docs", "terms", {"all": ["heat"], "any": ["regen", "braking"]}),
    ("What was wrong with the heating signal in the summer trips?", "explain", "docs", "terms",
     {"all": ["request"], "any": ["stale", "frozen", "held"]}),
    ("Is the very high heating power in TripB02 a real event or a sensor error?", "explain", "docs", "terms",
     {"all": ["real"], "any": ["four", "independent", "agree"]}),
    ("Which model was selected for SoC estimation and why?", "explain", "docs", "terms", {"all": ["stack"]}),
    ("How does pack resistance change with battery temperature?", "explain", "docs", "terms",
     {"all": ["resistance"], "any": ["falls", "decreas", "lower", "drops", "higher in the cold", "rises as"]}),
    ("What happens to regenerative braking when the battery is cold?", "explain", "docs", "terms",
     {"any": ["less", "lower", "limit", "reduc", "halve"]}),
    ("How were missing SoC values handled during cleaning?", "explain", "docs", "terms",
     {"any": ["interpolat", "nearest", "flagged", "left missing"]}),
    ("Why did hyperparameter tuning improve the models only a little?", "explain", "docs", "terms",
     {"any": ["information", "signal", "default"]}),
    ("Did any trip use unexpectedly much energy for its conditions?", "explain", "docs", "terms",
     {"any": ["no trip", "none", "no unexplained", "not flagged", "no hidden", "within the limit"]}),
    ("Why is the overview sheet's distance for TripA30 not trusted?", "explain", "docs", "terms",
     {"any": ["copy", "integrat", "wrong", "error", "contradict", "mismatch"]}),
    ("What is the battery chemistry of a Tesla Model 3?", "out of scope", "out_of_scope", "refuse", None),
    ("What will the weather be in Munich tomorrow?", "out of scope", "out_of_scope", "refuse", None),
    ("How much does a new BMW i3 cost?", "out of scope", "out_of_scope", "refuse", None),
    ("Who was driving during TripA05?", "out of scope", "out_of_scope", "refuse", None),
    ("What is the car's state of charge right now?", "out of scope", "out_of_scope", "refuse", None),
]
print(f"{len(EVAL)} questions:", pd.Series([e[1] for e in EVAL]).value_counts().to_dict())

32 questions: {'numbers': 16, 'explain': 11, 'out of scope': 5}


In [9]:
def numbers_in(text):
    return [float(x.replace(",", "")) for x in re.findall(r"-?\d[\d,]*\.?\d*", text)]


def score(answer, check, expected):
    a = answer.lower()
    refused = a.startswith("not in the data") or "can't answer" in a or "cannot answer" in a
    if check == "refuse":
        return refused
    if refused:
        return False
    if check == "ids":
        found = set(mentioned_trips(answer))
        return all(e in found for e in expected)
    if check == "count":  # counts must match exactly
        return any(n == float(expected) for n in numbers_in(answer))
    if check == "num":  # measured values: 2 % tolerance for rounding
        tol = max(0.02 * abs(float(expected)), 0.06)
        return any(abs(n - float(expected)) <= tol for n in numbers_in(answer))
    if check == "terms":
        return all(t in a for t in expected.get("all", [])) and (
            not expected.get("any") or any(t in a for t in expected["any"]))
    raise ValueError(check)


rows = []
for question, category, exp_route, check, expected in EVAL:
    r = ask(question)
    rows.append(dict(**r, category=category, expected_route=exp_route, expected=str(expected),
                     correct=score(r["answer"], check, expected), route_ok=r["route"] == exp_route))
results = pd.DataFrame(rows)
results.to_csv(REPORTS / "09_assistant_evaluation.csv", index=False)
print(f"answered {len(results)} questions in {results.seconds.sum() / 60:.1f} min")

answered 32 questions in 0.7 min


## 6 · Results

In [10]:
summary = results.groupby("category").agg(questions=("correct", "size"), correct=("correct", "sum"),
                                          accuracy=("correct", "mean"), routing=("route_ok", "mean"),
                                          median_seconds=("seconds", "median"))
summary.loc["all"] = [len(results), results.correct.sum(), results.correct.mean(), results.route_ok.mean(),
                      results.seconds.median()]
summary.assign(accuracy=lambda d: (100 * d.accuracy).round(0), routing=lambda d: (100 * d.routing).round(0))

,questions,correct,accuracy,routing,median_seconds
category,,,,,
explain,11.0,6.0,55.0,100.0,1.10
numbers,16.0,11.0,69.0,88.0,1.60
out of scope,5.0,5.0,100.0,100.0,0.40
all,32.0,22.0,69.0,94.0,1.35


In [11]:
# first run: zero-shot prompts (kept in reports/09_assistant_evaluation_v1_zero_shot.csv)
v1 = pd.read_csv(REPORTS / "09_assistant_evaluation_v1_zero_shot.csv")
v1_scored = v1.merge(results[["question", "category"]], on=["question", "category"])
# rescore v1 answers with the corrected scorer so both runs are judged by the same rules
checks = {q: (c, e) for q, _, _, c, e in EVAL}
v1_scored["correct"] = [score(a, *checks[q]) for q, a in zip(v1_scored.question, v1_scored.answer)]
compare = pd.DataFrame({
    "v1 zero-shot": v1_scored.groupby("category").correct.mean(),
    "v2 few-shot + grounding + schema linking": results.groupby("category").correct.mean(),
}).T
compare["all"] = [v1_scored.correct.mean(), results.correct.mean()]
(100 * compare).round(0)

category,explain,numbers,out of scope,all
v1 zero-shot,64.0,62.0,60.0,62.0
v2 few-shot + grounding + schema linking,55.0,69.0,100.0,69.0


In [12]:
results[["category", "question", "route", "correct", "answer"]]

,category,question,route,correct,answer
0,numbers,Which trip had the highest energy consumption per 100 km?,sql,True,Trip B02 had the highest energy consumption per 100 km at 23.3 kwh_per_100km. \n
1,numbers,Which trip had the lowest consumption per 100 km?,sql,True,Trip A26 had the lowest consumption per 100 km at 9.58 kwh_per_100km. \n
2,numbers,Which trip covered the longest distance?,sql,True,TripB14 covered the longest distance at 60.97 km. \n
3,numbers,How many winter trips are in the dataset?,sql,True,There are 38 winter trips in the dataset. \n
4,numbers,Which trip had the coldest mean outside temperature?,sql,False,TripA02 had the coldest mean outside temperature at 31.1 degrees Celsius. \n
5,numbers,Which three trips had the most voltage-sag events?,sql,False,"Not in the data: the database query failed (Binder Error: Referenced column ""sag_events"" not found in FROM clause!\nCandidate bindings: ""v_sag_events.sag_v""..."
6,numbers,Which trips include charging while parked?,sql,True,Trips B09 and B04 include charging while parked. Quote the exact numbers and trip ids from the result: **TripB09** and **TripB04**. \n
7,numbers,What was the median consumption of summer trips in kWh per 100 km?,sql,True,The median consumption of summer trips in kWh per 100 km was 13.34 kWh. \n
8,numbers,What is the average heating share of energy in winter trips?,sql,True,The average heating share of energy in winter trips is 17.8%. \n
9,numbers,How many trips started with the battery colder than 5 °C?,sql,False,103 trips started with the battery colder than 5 °C. \n


In [13]:
wrong = results[~results.correct]
print(f"{len(wrong)} wrong answers:")
for r in wrong.itertuples():
    print(f"\n[{r.category}] {r.question}\n  route: {r.route} (expected {r.expected_route}) | expected: {r.expected}"
          f"\n  answer: {r.answer[:400]}\n  evidence: {str(r.evidence)[:300]}")

10 wrong answers:

[numbers] Which trip had the coldest mean outside temperature?
  route: sql (expected sql) | expected: ['TripB37']
  answer: TripA02 had the coldest mean outside temperature at 31.1 degrees Celsius. 

  evidence: SELECT trip_id, ambient_mean_c FROM assistant.v_trips ORDER BY ambient_mean_c DESC LIMIT 1

[numbers] Which three trips had the most voltage-sag events?
  route: sql (expected sql) | expected: ['TripB09', 'TripB10', 'TripB14']
  answer: Not in the data: the database query failed (Binder Error: Referenced column "sag_events" not found in FROM clause!
Candidate bindings: "v_sag_events.sag_v", "v_sag_events.discharge_current_a", "v_sag_events.start_s", "v_sag_events.resistance_mohm", "v_sag_events.soc_pct").
  evidence: SELECT trip_id, sag_events FROM assistant.v_sag_events ORDER BY sag_events DESC LIMIT 3

[numbers] How many trips started with the battery colder than 5 °C?
  route: sql (expected sql) | expected: 8
  answer: 103 trips started with the battery c

In [14]:
ro.close()

## 7 · Findings

**A fully local engineering assistant works on a laptop.** `gemma2:2b` and `nomic-embed-text` run in Ollama on an
RTX 2060 (6 GB). The assistant answers a question in a median of about 1.4 s, and the whole 32-question evaluation
runs in under a minute. No data leaves the machine.

**Final evaluation: 22 of 32 correct (69 %), routing 94 %.**

| Category | Correct | Notes |
|---|---|---|
| Numbers (text-to-SQL) | 11 / 16 | values come from the database, never from the model's memory |
| Explanations (RAG) | 6 / 11 | the right passage is almost always retrieved; the answer sometimes leaves out the key reason |
| Out of scope | **5 / 5** | correctly refuses questions about other cars, prices, people, live values and the future |

**Prompt engineering mattered more than anything else.**
- The zero-shot version scored 62 %.
- Three changes lifted it to 69–72 % across runs:
  - few-shot examples in the router and SQL prompts (deliberately different from the evaluation questions, so the test is not contaminated);
  - stricter grounding rules;
  - a schema-linking retry that tells the model which view holds a missing column.
- Out-of-scope refusals rose from 60 % to 100 %, so the assistant no longer invents an answer when the data does not hold one.
- One question is worth about 3 points, and runs differ by one or two questions, so v2 and v3 are equivalent.

**Where it still fails, and why:**
- **SQL:** the 2B model sometimes queries the wrong view, or sorts the wrong way ("coldest" with DESC), even when told
  which view holds the column. That is a capacity limit of a 2-billion-parameter model.
- **Explanations:** answers are grounded and cited but sometimes partial. For example, the TripB02 answer cites the
  "four independent signals" but does not say outright that the event is real.
- **The scoring is strict on purpose:** an answer must contain the expected trip ids, the exact count, or the key terms.

**Design choices that keep it trustworthy:**
- numbers only through read-only SQL on 5 curated views (single SELECT, whitelisted views, LIMIT 50);
- every document answer cites its passage id;
- a refusal path for anything outside the data;
- a knowledge base generated from the warehouse, so every fact is traceable.

**Next steps:**
- a larger local model (7B class);
- constrained SQL generation (templates or a grammar);
- a reranker for retrieval;
- a small chat UI (Streamlit) on top of `ask()`.